# EfficientNet-B0 Transfer Learning Experiments

## Objective

The objective of this notebook is to evaluate EfficientNet-B0 for the screen recapture detection task.

EfficientNet-B0 is a lightweight convolutional neural network that balances model accuracy and computational efficiency through compound scaling.

The model is fine-tuned using transfer learning and evaluated with Stratified 5-Fold Cross Validation before being compared with MobileNetV3, ResNet18, and ResNet34.

## Import Required Libraries

Import all libraries required for transfer learning, optimization, dataset loading, evaluation, and cross-validation.

In [ ]:
import os
import copy
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score

## Define Image Preprocessing

Create the preprocessing pipelines for training and validation.

Training images are augmented to improve generalization, while validation images undergo deterministic preprocessing.

In [ ]:
train_transform = transforms.Compose([

    transforms.Resize((256,256)),

    transforms.RandomResizedCrop(
        224,
        scale=(0.85,1.0)
    ),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(8),

    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])



test_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

In [ ]:
dataset_path = "../dataset"

dataset = datasets.ImageFolder(dataset_path)

labels = dataset.targets

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

## Training Utilities

Define reusable training and evaluation functions used throughout the cross-validation experiment.

In [ ]:
def train_one_epoch(model, loader):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(outputs,1)

        total += labels.size(0)

        correct += (preds==labels).sum().item()

    epoch_loss = running_loss/len(loader)

    epoch_acc = correct/total

    return epoch_loss, epoch_acc

## Evaluation Utilities

Implement the validation procedure for computing prediction accuracy and collecting evaluation metrics after every training epoch.

In [ ]:
def evaluate(model, loader):

    model.eval()

    predictions = []

    targets = []

    correct = 0

    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            labels = labels.to(device)

            outputs = model(images)

            _, preds = torch.max(outputs,1)

            predictions.extend(preds.cpu().numpy())

            targets.extend(labels.cpu().numpy())

            total += labels.size(0)

            correct += (preds==labels).sum().item()

    accuracy = correct/total

    return accuracy,predictions,targets

## Train EfficientNet-B0

Fine-tune a pretrained EfficientNet-B0 model using Stratified 5-Fold Cross Validation.

Training includes:

- Transfer Learning
- AdamW Optimizer
- Label Smoothing
- Learning-Rate Scheduling
- Early Stopping
- Best Model Checkpoint Saving

In [ ]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_results=[]

best_fold_accuracy=0

best_fold=-1

os.makedirs("../models",exist_ok=True)

EPOCHS=30

PATIENCE=5

for fold,(train_idx,val_idx) in enumerate(
        skf.split(np.arange(len(dataset)),labels),1):

    print("="*60)
    print(f"Fold {fold}")
    print("="*60)

    train_dataset=copy.deepcopy(dataset)
    train_dataset.transform=train_transform

    val_dataset=copy.deepcopy(dataset)
    val_dataset.transform=test_transform

    train_subset=Subset(train_dataset,train_idx)
    val_subset=Subset(val_dataset,val_idx)

    train_loader=DataLoader(
        train_subset,
        batch_size=8,
        shuffle=True
    )

    val_loader=DataLoader(
        val_subset,
        batch_size=8,
        shuffle=False
    )
    weights=models.EfficientNet_B0_Weights.DEFAULT

    model=models.efficientnet_b0(
        weights=weights
    )

    for param in model.features.parameters():
        param.requires_grad=False

    for param in model.features[-2:].parameters():
        param.requires_grad=True

    model.classifier[1]=nn.Linear(
        1280,
        2
    )

    model=model.to(device)

    criterion=nn.CrossEntropyLoss(
        label_smoothing=0.05
    )

    optimizer=optim.AdamW(

        filter(
            lambda p:p.requires_grad,
            model.parameters()
        ),

        lr=1e-4,

        weight_decay=1e-4

    )

    scheduler=optim.lr_scheduler.ReduceLROnPlateau(

        optimizer,

        mode="max",

        factor=0.5,

        patience=2

    )

    best_acc=0

    counter=0

    best_model=copy.deepcopy(model.state_dict())


    for epoch in range(EPOCHS):

        loss,train_acc=train_one_epoch(
            model,
            train_loader
        )

        val_acc,preds,targets=evaluate(
            model,
            val_loader
        )

        scheduler.step(val_acc)

        print(
            f"Epoch {epoch+1:02d} | "
            f"Loss:{loss:.4f} | "
            f"Train:{train_acc:.4f} | "
            f"Val:{val_acc:.4f}"
        )

        if val_acc>best_acc:

            best_acc=val_acc

            best_model=copy.deepcopy(
                model.state_dict()
            )

            counter=0

        else:

            counter+=1

        if counter>=PATIENCE:

            print("Early Stopping...")
            break



    model.load_state_dict(best_model)

    val_acc,preds,targets=evaluate(
        model,
        val_loader
    )

    precision=precision_score(targets,preds)

    recall=recall_score(targets,preds)

    f1=f1_score(targets,preds)

    print(f"\nBest Validation Accuracy : {val_acc:.4f}")

    if val_acc>best_fold_accuracy:

        best_fold_accuracy=val_acc

        best_fold=fold

        torch.save(

            model.state_dict(),

            "../models/best_efficientnet_b0_screen_detector.pth"

        )

        print("Best model saved!")

    fold_results.append({

        "accuracy":val_acc,

        "precision":precision,

        "recall":recall,

        "f1":f1

    })

## Cross-Validation Results

Aggregate the evaluation metrics obtained from every validation fold and compute the mean Accuracy, Precision, Recall, and F1 Score for the EfficientNet-B0 architecture.

In [ ]:
acc=[x["accuracy"] for x in fold_results]

prec=[x["precision"] for x in fold_results]

rec=[x["recall"] for x in fold_results]

f1=[x["f1"] for x in fold_results]

print("="*60)

print("Cross Validation Results")

print("="*60)

print("Accuracy :",acc)

print()

print("Precision :",prec)

print()

print("Recall :",rec)

print()

print("F1 :",f1)

print()

print(f"Mean Accuracy : {np.mean(acc):.4f}")

print(f"Std Accuracy : {np.std(acc):.4f}")

print()

print(f"Mean Precision : {np.mean(prec):.4f}")

print(f"Mean Recall : {np.mean(rec):.4f}")

print(f"Mean F1 : {np.mean(f1):.4f}")

print()

print("="*60)

print(f"Best Fold : {best_fold}")

print(f"Best Accuracy : {best_fold_accuracy:.4f}")

print("../models/best_efficientnet_b0_screen_detector.pth")

print("="*60)

# Summary

This notebook evaluated EfficientNet-B0 using Stratified 5-Fold Cross Validation.

The architecture demonstrated competitive performance and achieved the highest single-fold validation accuracy among all evaluated models.

However, its average cross-validation performance remained slightly below that of ResNet34.

Consequently, ResNet34 was selected as the final deployment model because it provided the most consistent performance across all validation folds while maintaining fast inference speed.